In [ ]:
# import
import os, sys, pickle
import numpy as np
import pandas as pd
import scipy as sp
from tqdm import tqdm

# import plotting libraries
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 8})
plt.rcParams["svg.fonttype"] = "none"
import seaborn as sns
from nilearn import datasets
from nilearn import plotting

sys.path.extend([r'/home/lindenmp/research_projects/snaplab_tools'])
sys.path.extend([r'/home/lindenmp/research_projects/nctpy/src'])

from src.ints import compute_timescales, get_task_events, compute_autocorrelation_function, compute_task_autocorr_analysis
from src.plotting import plot_autocorrelation_analysis

In [ ]:
from nilearn.plotting import plot_design_matrix

import warnings
warnings.simplefilter('ignore', UserWarning)

## Setup

In [ ]:
# directory where data is stored
indir = '/home/lindenmp/research_projects/intrinsic_neural_timescales/data'
which_data = 'HCP-YA'
# which_data = 'HCP-D'
# which_data = 'RBC-PNC'
# which_data = 'RBC-HBN'
# which_data = 'RBC-NKI'
outdir = '/home/lindenmp/research_projects/intrinsic_neural_timescales/results/{0}'.format(which_data.replace('_', ''))

if not os.path.exists(outdir):
    os.makedirs(outdir)

atlas = 'Schaefer4007'
if atlas == 'Schaefer4007':
    n_parcels = 400
    n_nodes = 400
elif atlas == 'Schaefer2007':
    n_parcels = 200
    n_nodes = 200
elif atlas == 'Schaefer1007':
    n_parcels = 100
    n_nodes = 100
    
if which_data == 'HCP-YA':
    tr = 0.720
    rest_scans = ['rfMRI_REST1_LR', 'rfMRI_REST2_LR', 'rfMRI_REST1_RL', 'rfMRI_REST2_RL']
    tasks = ['tfMRI_WM_LR', 'tfMRI_EMOTION_LR', 'tfMRI_GAMBLING_LR', 'tfMRI_LANGUAGE_LR', 'tfMRI_RELATIONAL_LR', 'tfMRI_SOCIAL_LR',
             'tfMRI_WM_RL', 'tfMRI_EMOTION_RL', 'tfMRI_GAMBLING_RL', 'tfMRI_LANGUAGE_RL', 'tfMRI_RELATIONAL_RL', 'tfMRI_SOCIAL_RL']
elif which_data == 'HCP-D':
    tr = 0.800
    rest_scans = ['rfMRIREST1AP', 'rfMRIREST1PA', 'rfMRIREST2AP', 'rfMRIREST2PA']
    # tasks = ['tfMRI_CARIT_PA', 'tfMRI_EMOTION_PA', 'tfMRI_GUESSING_PA']
    tasks = ['tfMRI_CARIT_PA', 'tfMRI_GUESSING_PA']
    # tasks = ['tfMRI_CARIT_PA',]
elif which_data == 'RBC-PNC':
    tr = 3
    tasks = ['frac2back',]
elif which_data == 'RBC-HBN':
    tr = 0.800
    # tr_rutgers = 0.800
    # tr_staten = 1.45
elif which_data == 'RBC-NKI':
    tr = 0.800

In [ ]:
subject_ids = np.loadtxt(os.path.join(indir, '{0}_{1}_subjids.txt'.format(which_data, atlas)), dtype=str)
n_subs = len(subject_ids)
print(n_subs)

## Resting-state timescales

In [ ]:
# rest_timescales = dict()
# for i, which_scan in enumerate(rest_scans[:1]):
#     print(which_scan)
#     rest_ts = np.load(os.path.join(indir, '{0}_{1}_rsts-{2}.npy'.format(which_data, atlas, which_scan.replace('_', ''))))
#     print(rest_ts.shape)

#     rest_timescales[which_scan] = compute_timescales(rest_ts, return_mean=False).T
#     print(rest_timescales[which_scan].shape)

In [ ]:
rest_timescales = dict()
rest_timescales_auc = dict()

for which_scan in rest_scans:
    print(which_scan)
    rest_ts = np.load(os.path.join(indir, '{0}_{1}_rsts-{2}.npy'.format(which_data, atlas, which_scan.replace('_', ''))))
    print(rest_ts.shape)
    
    zero_crossings = np.zeros((n_subs, n_nodes))
    auc = np.zeros((n_subs, n_nodes))
    for i in tqdm(np.arange(n_subs)):
        _, zero_crossings[i, :], auc[i, :] = compute_autocorrelation_function(rest_ts[:, :, i].T)
    rest_timescales[which_scan] = zero_crossings
    rest_timescales_auc[which_scan] = auc

    print(rest_timescales[which_scan].shape, rest_timescales_auc[which_scan].shape)

## Task-based timescales and constrasts

In [ ]:
subject_filter_global = np.zeros(n_subs).astype(bool)
task_timescales = dict()
task_timescales_auc = dict()
contrasts = dict()

for which_task in tasks:
    print(which_task)
    if 'tfMRI_WM' in which_task:
        contrasts_list = ['0bk', '2bk']
    elif 'tfMRI_EMOTION' in which_task:
        contrasts_list = ['fear', 'neut']
    elif 'tfMRI_GAMBLING' in which_task:
        contrasts_list = ['win', 'loss']
    elif 'tfMRI_LANGUAGE' in which_task:
        contrasts_list = ['math', 'story']
    elif 'tfMRI_RELATIONAL' in which_task:
        contrasts_list = ['match', 'relation']
    elif 'tfMRI_SOCIAL' in which_task:
        contrasts_list = ['mental', 'rnd']
    elif 'tfMRI_CARIT' in which_task:
        contrasts_list = ['go', 'miss', 'nogoCRLose', 'nogoCRWin', 'nogoFALose', 'nogoFAWin']
        contrasts_list_short = ['go', 'nogo']
    elif 'tfMRI_EMOTION' in which_task:
        contrasts_list = ['faces', 'shapes']
    elif 'tfMRI_GUESSING' in which_task:
        contrasts_list = ['cueHigh', 'cueLow', 'feedbackHighLose', 'feedbackHighWin', 'feedbackLowLose', 'feedbackLowWin', 'guess']
    elif which_task == 'frac2back':
        contrasts_list = ['0back', '1back', '2back']

    subject_filter = np.zeros(n_subs).astype(bool)
    task_ts = np.load(os.path.join(indir, '{0}_{1}_taskts-{2}.npy'.format(which_data.replace('_', ''), atlas, which_task.replace('_', ''))))

    df_task_events_summary = pd.DataFrame(index=subject_ids)
    task_timescales[which_task] = np.zeros((n_subs, n_parcels))
    task_timescales_auc[which_task] = np.zeros((n_subs, n_parcels))

    if which_task == 'tfMRI_CARIT_PA':
        contrasts[which_task] = dict.fromkeys(contrasts_list_short)
        for contrast in contrasts_list_short:
            contrasts[which_task][contrast] = np.zeros((n_subs, n_parcels))
    else:
        contrasts[which_task] = dict.fromkeys(contrasts_list)
        for contrast in contrasts_list:
            contrasts[which_task][contrast] = np.zeros((n_subs, n_parcels))

    for i in tqdm(np.arange(n_subs)):
        try:
            subjid = subject_ids[i]
            if which_data == 'HCP-YA':
                ev_dir = os.path.join('/mnt/storage_ssd_raid/research_data/HCP_YA/HCP_1200', str(subjid), 'MNINonLinear', 'Results', which_task, 'EVs')
                task_events = get_task_events(ev_dir)
            elif which_data == 'HCP-D':
                subjid_converted = subjid.split('-')[1].replace('V1MR', '_V1_MR')
                ev_dir = os.path.join('/mnt/storage_ssd_raid/research_data/HCP_D/processed', subjid_converted, 'MNINonLinear', 'Results', which_task, 'EVs')
                task_events = get_task_events(ev_dir)
            elif which_data == 'RBC-PNC':
                ev_dir = os.path.join('/mnt/storage_ssd_raid/research_data/RBC/PNC/bids', subjid, 'ses-PNC1', 'func')
                ev_file = os.path.join(ev_dir, '{0}_ses-PNC1_task-frac2back_events.tsv'.format(subjid))
                task_events = pd.read_csv(ev_file, delimiter='\t')

            if which_task == 'tfMRI_CARIT_PA':
                for j in np.arange(task_events.shape[0]):
                    if 'nogo' in task_events.loc[j, 'trial_type']:
                        task_events.loc[j, 'trial_type'] = 'nogo'
                
                retain_idx = task_events['trial_type'] != 'miss'
                task_events = task_events.loc[retain_idx, :]
            elif which_task == 'tfMRI_WM_LR' or which_task == 'tfMRI_WM_RL':
                for j in np.arange(task_events.shape[0]):
                    if '0bk' in task_events.loc[j, 'trial_type']:
                        task_events.loc[j, 'trial_type'] = '0bk'
                    elif '2bk' in task_events.loc[j, 'trial_type']:
                        task_events.loc[j, 'trial_type'] = '2bk'
            # elif which_task == 'frac2back':
                # task_events = task_events[task_events['trial_type'] != 'instr']
                # task_events.reset_index(drop=True, inplace=True)
                
            # unique_events = np.unique(task_events['trial_type'])
            for event in list(contrasts[which_task].keys()):
                df_task_events_summary.loc[subjid, event] = np.sum(task_events['trial_type'] == event)

            fmri_data = task_ts[:, :, i].T
            results = compute_task_autocorr_analysis(fmri_data, task_events, tr, hrf_model='glover', drift_model='cosine')
            task_timescales[which_task][i, :] = results['zero_crossings_resid']
            task_timescales_auc[which_task][i, :] = results['auc_resid']
            
            # compute contrasts    
            beta = np.dot(np.linalg.pinv(results['design_matrix'].values), fmri_data.T)
            for contrast in list(contrasts[which_task].keys()):
                contrast_idx = list(results['design_matrix'].columns).index(contrast)
                contrasts[which_task][contrast][i, :] = beta[contrast_idx, :]
                
            if i == 0:
                print(np.unique(task_events['trial_type']))
                plot_design_matrix(results['design_matrix'])
                plt.show()
            if i % 100 == 0:
                outdir_plot = os.path.join(outdir, 'plot_autocorrelation_analysis')
                if not os.path.exists(outdir_plot):
                    os.makedirs(outdir_plot)
                plot_autocorrelation_analysis(results['autocorr_raw'], results['autocorr_resid'], 
                                              results['zero_crossings_raw'], results['zero_crossings_resid'], results['design_matrix'],
                                              task_events, tr, save_path=os.path.join(outdir_plot, f'{subjid}_{which_task}.png'))
                plt.close('all')
        except Exception as e:
            subject_filter[i] = True
            print(e)
    print(np.sum(subject_filter))
    subject_filter_global += subject_filter

    for contrast in list(contrasts[which_task].keys()):
        contrasts[which_task][contrast] = np.asarray(contrasts[which_task][contrast])

## Save out

In [ ]:
np.any(subject_filter_global)

In [ ]:
if np.any(subject_filter_global):
    subject_ids_filt = subject_ids[~subject_filter_global]
    print(subject_ids_filt.shape)

    for which_scan in rest_timescales.keys():
        rest_timescales[which_scan] = rest_timescales[which_scan][~subject_filter_global, :]
        print(rest_timescales[which_scan].shape)
        
        rest_timescales_auc[which_scan] = rest_timescales_auc[which_scan][~subject_filter_global, :]
        print(rest_timescales_auc[which_scan].shape)
        
    for which_task in task_timescales.keys():
        task_timescales[which_task] = task_timescales[which_task][~subject_filter_global, :]
        print(task_timescales[which_task].shape)

        task_timescales_auc[which_task] = task_timescales_auc[which_task][~subject_filter_global, :]
        print(task_timescales_auc[which_task].shape)

    for which_task in contrasts.keys():
        for contrast in contrasts[which_task].keys():
            contrasts[which_task][contrast] = contrasts[which_task][contrast][~subject_filter_global, :]
            print(contrasts[which_task][contrast].shape)
            
    np.savetxt(os.path.join(outdir, '{0}_{1}_subjids.txt'.format(which_data, atlas)), subject_ids_filt, fmt='%s')

In [ ]:
for which_scan in rest_timescales.keys():
    np.save(os.path.join(outdir, '{0}_{1}_{2}_timescales_acf-lag.npy'.format(which_data, atlas, which_scan.replace('_', ''))), rest_timescales[which_scan])
    np.save(os.path.join(outdir, '{0}_{1}_{2}_timescales_acf-auc.npy'.format(which_data, atlas, which_scan.replace('_', ''))), rest_timescales_auc[which_scan])

for which_scan in task_timescales.keys():
    np.save(os.path.join(outdir, '{0}_{1}_{2}_timescales_acf-lag.npy'.format(which_data, atlas, which_scan.replace('_', ''))), task_timescales[which_scan])
    np.save(os.path.join(outdir, '{0}_{1}_{2}_timescales_acf-auc.npy'.format(which_data, atlas, which_scan.replace('_', ''))), task_timescales_auc[which_scan])

    with open(os.path.join(outdir, '{0}_{1}_{2}_contrasts.pkl'.format(which_data, atlas, which_scan.replace('_', ''))), 'wb') as f:
        pickle.dump(contrasts[which_scan], f)